In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVM
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
import os

In [12]:
df = pd.read_csv("chat_dataset_quoted.csv")
df = df.dropna()

print(df['label_intent'].value_counts())
print(df['teks_chat'].duplicated().sum())

label_intent
accusing      102
persuading     98
defending      97
probing        97
deflecting     97
claiming       97
bluffing       96
neutral        96
Name: count, dtype: int64
1


In [14]:
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
    SVC(kernel='linear', probability=True, class_weight='balanced')  # penting kalau label imbalance
)

In [13]:
TfidfVectorizer(
    ngram_range=(1, 2),      # tangkap frasa "kamu gangster", bukan cuma kata tunggal
    min_df=2,                # buang kata yang muncul cuma 1x (biasanya typo/noise)
    max_df=0.9,              # buang kata yang muncul di hampir semua chat (kurang informatif)
    sublinear_tf=True,       # log-scale term frequency, bantu buat teks pendek
    token_pattern=r"(?u)\b\w+\b",  # default TF-IDF suka gagal capture kata 1 huruf/slang tertentu
)

TfidfVectorizer(max_df=0.9, min_df=2, ngram_range=(1, 2), sublinear_tf=True,
                token_pattern='(?u)\\b\\w+\\b')

In [16]:
def train_intent_model():
    print("Membaca dataset chat_dataset.csv...")
    dataset_path = "data/chat_dataset_quoted.csv"
    
    # Cek apakah dataset sudah ada
    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None
        
    # Load data menggunakan pandas
    df = pd.read_csv(dataset_path)
    
    # Bersihkan baris yang kosong (handling missing values)
    df = df.dropna()
    
    X = df['teks_chat']
    y = df['label_intent']
    
    # Bagi data: 80% untuk AI belajar, 20% untuk ujian (testing)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    
    # Membuat pipeline: TF-IDF (ubah teks jadi angka) -> SVM (klasifikasi intent)
    print("Melatih model NLU (TF-IDF + SVM)...")
    model = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', probability=True))
    
    # Proses Training
    model.fit(X_train, y_train)
    
    # Evaluasi Akurasi Model
    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    # Simpan model agar tidak perlu dilatih ulang setiap kali server menyala
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier.pkl")
    print("Model berhasil disimpan di models/intent_classifier.pkl\n")
    
    return model

In [19]:
import os
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC as SVM   # alias biar kelihatan "SVM" di kode
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report


def load_and_split_data(dataset_path="chat_dataset_quoted.csv",
                         test_size=0.2, random_state=42):
    """Load dataset, bersihkan, dan split jadi train/test."""
    print(f"Membaca dataset {dataset_path}...")

    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None

    df = pd.read_csv(dataset_path)
    df = df.dropna()

    print("Distribusi label:")
    print(df['label_intent'].value_counts())
    print(f"Jumlah duplikat teks: {df['teks_chat'].duplicated().sum()}\n")

    X = df['teks_chat']
    y = df['label_intent']

    return train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y  # penting biar distribusi label di test set representatif
    )


def tune_intent_model(X_train, y_train):
    """Cari kombinasi parameter terbaik pakai GridSearchCV."""
    pipeline = make_pipeline(
        TfidfVectorizer(),
        SVM(kernel='linear', probability=True, class_weight='balanced')
    )

    param_grid = {
        'tfidfvectorizer__ngram_range': [(1, 1), (1, 2)],
        'tfidfvectorizer__min_df': [1, 2, 3],
        'svc__C': [0.1, 1, 10],
    }

    print("Mencari parameter terbaik (GridSearchCV)...")
    grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1_macro')
    grid.fit(X_train, y_train)

    print(f"Parameter terbaik: {grid.best_params_}")
    print(f"Best CV f1_macro score: {grid.best_score_:.4f}\n")

    return grid.best_estimator_


def train_intent_model(model=None, use_tuning=True):
    """
    Latih model NLU intent classifier.
    - model=None & use_tuning=True  -> cari parameter terbaik otomatis (GridSearchCV)
    - model=None & use_tuning=False -> pakai pipeline default
    - model diisi (mis. hasil tuning manual) -> langsung dievaluasi & disimpan
    """
    split = load_and_split_data()
    if split is None:
        return None
    X_train, X_test, y_train, y_test = split

    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")

    if model is None:
        if use_tuning:
            model = tune_intent_model(X_train, y_train)
        else:
            print("Melatih model NLU (TF-IDF + SVM) dengan parameter default...")
            model = make_pipeline(
                TfidfVectorizer(),
                SVM(kernel='linear', probability=True, class_weight='balanced')
            )
            model.fit(X_train, y_train)

    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))

    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier.pkl")
    print("Model berhasil disimpan di models/intent_classifier.pkl\n")

    return model


if __name__ == "__main__":
    final_model = train_intent_model(use_tuning=True)

Membaca dataset chat_dataset_quoted.csv...
Distribusi label:
label_intent
accusing      102
persuading     98
defending      97
probing        97
deflecting     97
claiming       97
bluffing       96
neutral        96
Name: count, dtype: int64
Jumlah duplikat teks: 1

Total data latih: 624 | Total data uji: 156
Mencari parameter terbaik (GridSearchCV)...
Parameter terbaik: {'svc__C': 1, 'tfidfvectorizer__min_df': 1, 'tfidfvectorizer__ngram_range': (1, 2)}
Best CV f1_macro score: 0.8522


--- Hasil Ujian Model (Evaluasi) ---
              precision    recall  f1-score   support

    accusing       1.00      0.95      0.97        20
    bluffing       0.80      0.84      0.82        19
    claiming       0.56      0.50      0.53        20
   defending       0.74      0.89      0.81        19
  deflecting       0.90      1.00      0.95        19
     neutral       0.73      0.58      0.65        19
  persuading       0.81      0.85      0.83        20
     probing       0.95      0.90    

In [20]:
final_model = train_intent_model(use_tuning=True)

Membaca dataset chat_dataset_quoted.csv...
Distribusi label:
label_intent
accusing      102
persuading     98
defending      97
probing        97
deflecting     97
claiming       97
bluffing       96
neutral        96
Name: count, dtype: int64
Jumlah duplikat teks: 1

Total data latih: 624 | Total data uji: 156
Mencari parameter terbaik (GridSearchCV)...
Parameter terbaik: {'svc__C': 1, 'tfidfvectorizer__min_df': 1, 'tfidfvectorizer__ngram_range': (1, 2)}
Best CV f1_macro score: 0.8522


--- Hasil Ujian Model (Evaluasi) ---
              precision    recall  f1-score   support

    accusing       1.00      0.95      0.97        20
    bluffing       0.80      0.84      0.82        19
    claiming       0.56      0.50      0.53        20
   defending       0.74      0.89      0.81        19
  deflecting       0.90      1.00      0.95        19
     neutral       0.73      0.58      0.65        19
  persuading       0.81      0.85      0.83        20
     probing       0.95      0.90    

In [21]:
def predict_intent(chat_text):
    # Load model yang sudah pintar
    model_path = "models/intent_classifier.pkl"
    if not os.path.exists(model_path):
        model = train_intent_model()
    else:
        model = joblib.load(model_path)
    
    # Prediksi intent dari chat baru
    prediksi = model.predict([chat_text])[0]
    
    # Ambil nilai probabilitas/keyakinan model (dalam persentase)
    probabilitas = max(model.predict_proba([chat_text])[0]) * 100
    
    return prediksi, probabilitas

In [4]:
print(df['label_intent'].value_counts())
print(df['teks_chat'].duplicated().sum())  # cek duplikat

NameError: name 'df' is not defined

In [27]:
train_intent_model()
    
print("--- SIMULASI TESTING AI 1 DI DALAM GAME ---")
test_chats = [
    # "Jagain rumah gw pak pol, gw bayar mahal nih pake koin"
    "aeh aku gatau apa-apa, aku cuma orang biasa",
    # "Woy si Budi dari tadi diem aja, fix dia ketuanya"
]

for chat in test_chats:
    intent, prob = predict_intent(chat)
    print(f"Chat Player: '{chat}'")
    print(f" > AI 1 Menebak: [{intent.upper()}] (Tingkat Keyakinan: {prob:.2f}%)\n")

Membaca dataset chat_dataset_quoted.csv...
Distribusi label:
label_intent
accusing      102
persuading     98
defending      97
probing        97
deflecting     97
claiming       97
bluffing       96
neutral        96
Name: count, dtype: int64
Jumlah duplikat teks: 1

Total data latih: 624 | Total data uji: 156
Mencari parameter terbaik (GridSearchCV)...
Parameter terbaik: {'svc__C': 1, 'tfidfvectorizer__min_df': 1, 'tfidfvectorizer__ngram_range': (1, 2)}
Best CV f1_macro score: 0.8522


--- Hasil Ujian Model (Evaluasi) ---
              precision    recall  f1-score   support

    accusing       1.00      0.95      0.97        20
    bluffing       0.80      0.84      0.82        19
    claiming       0.56      0.50      0.53        20
   defending       0.74      0.89      0.81        19
  deflecting       0.90      1.00      0.95        19
     neutral       0.73      0.58      0.65        19
  persuading       0.81      0.85      0.83        20
     probing       0.95      0.90    